# Khởi tạo Môi trường Google Colab T4 & Đồng bộ Kho Lưu trữ

Cell bên dưới sẽ tự động thực hiện:
1. **Kết nối Google Drive** và tạo các thư mục lưu trữ (`feature_store`, `checkpoints`).
2. **Kiểm tra thư mục code cũ**: Nếu đã tồn tại thì tự động xóa sạch và clone bản mới nhất từ GitHub.
3. **Chuyển vào thư mục làm việc chuẩn** và đăng ký `sys.path` để nạp module `benchmarks` không bao giờ bị lỗi.
4. **Kiểm tra phần cứng GPU T4** và xác thực sự hiện diện của các thư mục `core/`, `models/`, `metrics/`.

In [5]:
import os
import sys
import shutil
from pathlib import Path

# -------------------------------------------------------------------------
# 1. Kết nối Google Drive & Khởi tạo Thư mục Lưu trữ
# -------------------------------------------------------------------------
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('[1/4] Đang kết nối Google Drive...')
        drive.mount('/content/drive')
    else:
        print('[1/4] Google Drive đã được kết nối!')
except ImportError:
    print('[1/4] Chạy trên môi trường Local (Bỏ qua Colab Drive Mount)')

DRIVE_DIR = '/content/drive/MyDrive/multimodal_lecture_benchmark'
os.makedirs(f'{DRIVE_DIR}/feature_store', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'[OK] Thư mục lưu trữ Drive: {DRIVE_DIR}')

# -------------------------------------------------------------------------
# 2. Kiểm tra, Xóa bản cũ và Clone bản mới nhất từ GitHub
# -------------------------------------------------------------------------
REPO_URL = 'https://github.com/multimodal-lecture-summarizer/multimodal-lecture-summarizer.git'
TARGET_ROOT = '/content/multimodal-lecture-summarizer'

%cd /content
if os.path.exists(TARGET_ROOT):
    print(f'[2/4] Phát hiện thư mục code cũ tại {TARGET_ROOT}, đang xóa sạch...')
    shutil.rmtree(TARGET_ROOT)
    print('[OK] Đã xóa bản cũ thành công.')

print(f'[2/4] Đang clone mã nguồn mới nhất từ GitHub...')
!git clone {REPO_URL} {TARGET_ROOT}

# -------------------------------------------------------------------------
# 3. Chuyển vào đúng thư mục dự án và Đăng ký sys.path
# -------------------------------------------------------------------------
PROJECT_DIR = f'{TARGET_ROOT}'
%cd {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Tránh xung đột thư viện OpenMP
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['HF_HOME'] = '/root/.cache/huggingface'

# -------------------------------------------------------------------------
# 4. Chẩn đoán Môi trường & Xác thực Cấu trúc benchmarks/
# -------------------------------------------------------------------------
print('\n' + '='*60)
print('[OK] HOÀN TẤT THIẾT LẬP MÔI TRƯỜNG!')
print(f'- Thư mục hiện tại (CWD): {os.getcwd()}')

benchmarks_path = Path(PROJECT_DIR) / 'benchmarks'
if benchmarks_path.exists():
    subdirs = [d.name for d in benchmarks_path.iterdir() if d.is_dir()]
    print(f'- Module benchmarks/ đã sẵn sàng: {subdirs}')
else:
    print('[CẢNH BÁO] Không tìm thấy thư mục benchmarks/!')

import torch
gpu_status = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Chỉ có CPU (Chưa bật GPU T4)'
print(f'- Phần cứng: {gpu_status}')
print('='*60)
print('Bây giờ bạn có thể mở và chạy bất kỳ notebook nào (01, 02, 03)!')


[1/4] Google Drive đã được kết nối!
[OK] Thư mục lưu trữ Drive: /content/drive/MyDrive/multimodal_lecture_benchmark
/content
[2/4] Phát hiện thư mục code cũ tại /content/multimodal-lecture-summarizer, đang xóa sạch...
[OK] Đã xóa bản cũ thành công.
[2/4] Đang clone mã nguồn mới nhất từ GitHub...
Cloning into '/content/multimodal-lecture-summarizer'...
remote: Enumerating objects: 1351, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 1351 (delta 61), reused 139 (delta 48), pack-reused 1167 (from 1)
Receiving objects: 100% (1351/1351), 7.19 MiB | 12.14 MiB/s, done.
Resolving deltas: 100% (686/686), done.
/content/multimodal-lecture-summarizer

[OK] HOÀN TẤT THIẾT LẬP MÔI TRƯỜNG!
- Thư mục hiện tại (CWD): /content/multimodal-lecture-summarizer
- Module benchmarks/ đã sẵn sàng: ['metrics', 'models', 'references', 'manifests', 'scripts', 'core']
- Phần cứng: Tesla T4
Bây giờ bạn có thể mở và chạy bất kỳ notebook nào (01,